# Fine-tune the PLABA Simplifier (Mistral-7B + QLoRA)

Run this on **Google Colab** with a **GPU runtime** (`Runtime → Change runtime type → T4 GPU`).

Steps:
1. Install libraries **(this auto-restarts the runtime once — that's expected; just continue from step 2 afterwards, do NOT re-run step 1)**
2. Check the GPU
3. Upload the prepared data (`train.sft.jsonl`, `val.sft.jsonl`)
4. (Mistral/Llama are gated) log in to Hugging Face
5. Train the QLoRA adapter
6. Test the model
7. Download the adapter

> Prepare the data locally first with `python training/prepare_plaba_sft.py`, then upload the JSONL files in step 3.

In [ ]:
# 1) Install libraries (pinned to a matched, compatible set)
# Colab ships an older `transformers`, which breaks the newer `trl`. We pin a
# known-good combination so the imports line up.
!pip install -q -U \
    "transformers==4.55.2" \
    "trl==0.21.0" \
    "peft==0.15.2" \
    "accelerate==1.8.1" \
    "datasets==3.6.0" \
    "bitsandbytes==0.46.1" \
    sentencepiece

# IMPORTANT: restart the runtime ONCE after this install so Python drops the old
# transformers it loaded at startup. This cell does it automatically.
import os
os.kill(os.getpid(), 9)

In [ ]:
# 2) Check the GPU
import torch
assert torch.cuda.is_available(), "No GPU! Set Runtime -> Change runtime type -> T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))
print("bf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
# 3) Upload the prepared data: train.sft.jsonl and val.sft.jsonl
from google.colab import files
print("Select train.sft.jsonl and val.sft.jsonl from your computer...")
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

In [ ]:
# 4) Choose the base model and (optionally) log in to Hugging Face
#
# EASIEST: Qwen2.5-7B-Instruct is fully open -> no license form, no token needed.
# Mistral/Llama are GATED -> you must accept the license on the model page AND log in.
#
# Pick ONE:
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"          # ungated, recommended
# BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"  # gated: needs license + token

OUTPUT_DIR = "plaba-simplifier-qlora"

# Login is ONLY needed for gated models (Mistral/Llama). For Qwen you can skip it.
# The login widget sometimes doesn't render in Colab, so we read the token from
# Colab Secrets (left sidebar -> key icon -> add HF_TOKEN, enable Notebook access).
if "mistral" in BASE_MODEL.lower() or "llama" in BASE_MODEL.lower():
    from huggingface_hub import login
    try:
        from google.colab import userdata
        login(token=userdata.get("HF_TOKEN"))
        print("Logged in via Colab secret HF_TOKEN.")
    except Exception:
        import getpass
        login(token=getpass.getpass("Paste your HF token (hf_...): "))
else:
    print(f"{BASE_MODEL} is ungated - no Hugging Face login needed.")

In [ ]:
# 5) Train the QLoRA adapter
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if bf16 else torch.float16

dataset = load_dataset(
    "json",
    data_files={"train": "train.sft.jsonl", "validation": "val.sft.jsonl"},
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
)
model.config.use_cache = False

peft_config = LoraConfig(
    r=32,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    max_length=1024,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=bf16,
    fp16=not bf16,
    optim="paged_adamw_8bit",
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    peft_config=peft_config,
)
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Adapter saved to", OUTPUT_DIR)

In [ ]:
# 6) Quick test on one example
SYSTEM = ("You are a medical text simplifier writing for a general (lay) audience. "
          "Rewrite the source text into plain language while preserving all medical "
          "meaning. Use common words and short sentences, prefer active voice, define "
          "acronyms on first use, do not omit important facts, and do not add "
          "unsupported facts.")
source = ("Hyperkalemia is a frequent clinical abnormality in patients with chronic "
          "kidney disease, and it is associated with higher risk of mortality and "
          "malignant arrhythmias.")

msgs = [{"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Source text:\n{source}"}]
inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
out = model.generate(inputs, max_new_tokens=256, do_sample=True, temperature=0.3, top_p=0.9)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

In [ ]:
# 7) Download the trained adapter (zip it, then save to your computer)
import shutil
from google.colab import files
shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
files.download(f"{OUTPUT_DIR}.zip")

## After training

1. Unzip the downloaded adapter into `outputs/plaba-mistral-qlora/` in your local repo.
2. Merge + export for Ollama:
   ```bash
   python training/merge_and_export.py --adapter-dir outputs/plaba-mistral-qlora --merged-dir outputs/plaba-merged
   ```
3. Convert to GGUF (llama.cpp) and register with Ollama, then run the app:
   ```powershell
   $env:USE_OLLAMA_SIMPLIFIER = "1"
   python main.py --text "Hyperkalemia is a frequent clinical abnormality..."
   ```

See `training/README.md` for full details.